# Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path
from hydra.utils import instantiate
import torch

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.utils.notebook_setup import init_nlp_notebook # noqa: E402
cfg = init_nlp_notebook()
device = "cuda" if torch.cuda.is_available() else "cpu"

# Data init and collator

In [ ]:
tokenizer = instantiate(cfg.model.tokenizer).build()
datamodule = instantiate(cfg.data.module, tokenizer=tokenizer)
datamodule.prepare_data()
datamodule.setup(stage="validate")
val_dataloader = datamodule.val_dataloader()
sample_batch = next(iter(val_dataloader))

print("Keys in batch:", sample_batch.keys())
print("Input IDs shape:", sample_batch["input_ids"].shape)
if "labels" in sample_batch:
    print("Labels shape:", sample_batch["labels"].shape)

# Model init

In [ ]:
from src.core.models.custom_heads import MultiTaskBERT

# В Гидре это собирается через instantiate, но здесь мы можем показать логику явно:
base_builder = instantiate(cfg.model.architecture.base_builder)

# Твой класс принимает базовый билдер и количество классов для каждой головы
model = MultiTaskBERT(
    base_builder=base_builder,
    num_sentiment_classes=3,
    num_category_classes=5
)
model.to(device)
model.eval()

print(f"Model architecture: {model.__class__.__name__}")

# Forward Pass

In [ ]:
sample_batch = {k: v.to(device) for k, v in sample_batch.items() if isinstance(v, torch.Tensor)}

with torch.no_grad():
    # Проверка кастомного forward()[cite: 12]
    outputs = model(
        input_ids=sample_batch["input_ids"], 
        attention_mask=sample_batch["attention_mask"]
    )

print("Output keys:", outputs.keys())
print("Sentiment logits shape:", outputs["sentiment"].shape)
print("Category logits shape:", outputs["category"].shape)

# Проверка инициализации твоего LightningModule (NLPModel)[cite: 10]
lightning_module = instantiate(
    cfg.training.lightning_module, 
    model=model,
    num_classes=3 # Для инициализации MulticlassAccuracy/F1[cite: 10]
)
print(f"Metrics initialized: {lightning_module.val_metrics}")

# Baseline Evaluation

In [ ]:
from sklearn.metrics import classification_report
from tqdm.auto import tqdm
import numpy as np

all_preds = []
all_labels = []

# Ограничимся 50 батчами для быстрой проверки
max_batches = 50 

for i, batch in enumerate(tqdm(val_dataloader, desc="Baseline Eval", total=min(len(val_dataloader), max_batches))):
    if i >= max_batches:
        break
        
    batch = {k: v.to(device) for k, v in batch.items()}
    
    with torch.no_grad():
        outputs = model(**batch)
        
    preds = torch.argmax(outputs.logits, dim=-1)
    
    all_preds.extend(preds.cpu().numpy())
    all_labels.extend(batch["labels"].cpu().numpy())

print("\n--- UNTRAINED BASELINE REPORT ---")
print(classification_report(all_labels, all_preds, zero_division=0))

# Ожидание: Accuracy должно быть около (1 / количество_классов).
# Если Accuracy сильно выше случайного угадывания — возможно, произошла утечка данных (Data Leakage).